In [1]:
!pip install pdfplumber

In [25]:
import os
import pdfplumber
from google import genai
from pydantic import BaseModel
from typing import List
from getpass import getpass

# 1. Define Structured Schema
class Experience(BaseModel):
    role: str
    company: str
    description: str

class ResumeData(BaseModel):
    skills: List[str]
    experience: List[Experience]
    achievements: List[str]
    projects: List[str]
    github: str
    linkedin: str

def extract_raw_text(pdf_path: str) -> str:
    """Extracts raw text from PDF for display purposes."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return text.strip()

def parse_resume_gemini(pdf_path: str, api_key: str):
    client = genai.Client(api_key=api_key)

    # Uploading the file directly ensures Gemini uses its own OCR
    resume_file = client.files.upload(file=pdf_path)

    prompt = "Extract the structured information from this resume following the provided schema."

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite-preview", # Fixed model name
        contents=[resume_file, prompt],
        config={
            "response_mime_type": "application/json",
            "response_schema": ResumeData,
        }
    )
    return response.parsed
# --- Execution Flow ---
FILE_PATH = "/content/resume.pdf"
API_KEY = getpass("Enter your Google API Key: ")
# Step 1: Show Raw Extraction
print("--- PHASE 1: RAW TEXT EXTRACTION ---")
raw_text = extract_raw_text(FILE_PATH)
print(raw_text if raw_text else "No text found.")
print("\n" + "="*50 + "\n")

# Step 2: Show Structured Extraction
print("--- PHASE 2: STRUCTURED JSON OUTPUT ---")
structured_data = parse_resume_gemini(FILE_PATH, API_KEY)

# Displaying the structured data nicely
import json
print(json.dumps(structured_data.model_dump(), indent=2))

Enter your Google API Key: ··········


--- PHASE 1: RAW TEXT EXTRACTION ---


Manukrishna Sasidharan Nambiar
Date of birth: 24/11/2005
Address: Souparnika, Gopal Street, Kannur, 670001, Kerala, India
Phone number: +91 9995340964 Email address: manukrishnasasidharan@gmail.com
Profile
I am a tech enthusiast with a passion for programming, embedded systems and machine learning. I enjoy building
innovative projects that solve real-world problems. I am always eager to learn and explore new technologies.
EDUCATION AND TRAINING
08/2023 – present Electronics & Communication Engineering | BTech
Kerala, India National Institute of Technology Calicut
Final grade: 7.16
04/2022 – 04/2023 Class 12th
Dubai, The Millennium School
United Arab Emirates Final grade: 90.8%
04/2020 – 04/2021 Class 10th
Dubai, The Millennium School
United Arab Emirates Final grade: 95.4%
WORK EXPERIENCE
05/2025 – 06/2025 Pontem Analytics
London, United Kingdom INTERN
Internship Project: Condition Monitoring of Hydraulic Test Rig using Machine Learning
Designed and implemented an automated ML system t

Embedding Function

In [3]:
from google import genai

def get_gemini_embedding(text: str, api_key: str):
    client = genai.Client(api_key=api_key)

    # text-embedding-004 is the reliable free-tier model
    response = client.models.embed_content(
        model="gemini-embedding-2-preview",
        contents=text
    )

    # Gemini returns the embedding inside a 'values' list
    return response.embeddings[0].values

# Usage
# vector = get_gemini_embedding("Software Engineer with 5 years experience", "AIza...")
# print(f"Vector Length: {len(vector)}") # Output: 768

Cosine Similarity

In [4]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

Exact Match Score

In [5]:
def exact_match_score(resume_skills, jd_skills):
    # .strip() handles accidental spaces; if s filter handles empty strings
    resume_set = {s.strip().lower() for s in resume_skills if s}
    jd_set = {s.strip().lower() for s in jd_skills if s}

    if not jd_set:
        return 1.0  # If no skills are required, it's a perfect match by default

    matches = resume_set.intersection(jd_set)
    return len(matches) / len(jd_set)

Semantic Similarity Score

In [6]:
import numpy as np

def calculate_similarity_score(resume_embeddings: list, jd_embeddings: list) -> float:
    """
    Calculates semantic alignment.
    Expects lists of vectors (each 768-dim) from your get_gemini_embedding function.
    """
    if not resume_embeddings or not jd_embeddings:
        return 0.0

    # 1. Convert lists to Numpy matrices
    r_matrix = np.array(resume_embeddings) # Shape: (num_resume_skills, 768)
    j_matrix = np.array(jd_embeddings)     # Shape: (num_jd_skills, 768)

    # 2. Normalize vectors (Unit length)
    # This ensures the dot product equals the Cosine Similarity
    r_matrix /= np.linalg.norm(r_matrix, axis=1, keepdims=True)
    j_matrix /= np.linalg.norm(j_matrix, axis=1, keepdims=True)

    # 3. Compute Similarity Matrix
    # Dot product calculates similarity between every JD skill and every Resume skill
    sim_matrix = np.dot(j_matrix, r_matrix.T) # Shape: (num_jd_skills, num_resume_skills)

    # 4. Extract Best Matches
    # For each JD skill, find the highest similarity found in the resume
    best_matches = np.max(sim_matrix, axis=1)

    # 5. Aggregate Score
    # Return the average 'coverage' of the JD requirements
    return float(np.mean(best_matches))

# --- Example Workflow ---
# resume_vectors = [get_gemini_embedding(s, key) for s in parsed_resume.skills]
# jd_vectors = [get_gemini_embedding(s, key) for s in jd_skills]
# final_score = calculate_similarity_score(resume_vectors, jd_vectors)

Achievement Score

In [7]:
def achievement_score(experience: list) -> float:
    # Action verbs and quantifiers indicating high-impact work
    impact_indicators = {"%", "improved", "increased", "reduced", "scaled", "optimized", "saved", "launched"}

    if not experience:
        return 0.0

    scored_roles = 0
    for exp in experience:
        # Check if exp is a Pydantic object (has attribute) or a dict (standard)
        if hasattr(exp, "description"):
            desc = exp.description.lower()
        else:
            desc = exp.get("description", "").lower()

        if any(indicator in desc for indicator in impact_indicators):
            scored_roles += 1

    return scored_roles / len(experience)

Ownership Score

In [8]:
def ownership_score(experience):
    strong = ["led", "built", "designed", "architected", "spearheaded", "developed"]
    weak = ["assisted", "helped", "supported", "contributed"]

    if not experience:
        return 0.0

    score = 0
    for exp in experience:
        # Check if item is a Pydantic object or a dictionary
        if hasattr(exp, "description"):
            desc = exp.description.lower()
        else:
            desc = exp.get("description", "").lower()

        if any(w in desc for w in strong):
            score += 1
        elif any(w in desc for w in weak):
            score += 0.3

    return min(score / len(experience), 1.0)

Evaluation

In [9]:
def evaluate_candidate_optimized(resume_data, jd, api_key):
    # Convert Pydantic object to list for scoring
    resume_skills = resume_data.skills
    experience = resume_data.experience
    jd_skills = jd.get("skills", [])

    # 1. Exact Match (Keyword)
    exact = exact_match_score(resume_skills, jd_skills)

    # 2. Semantic Similarity (Optimized Batch)
    # Generate embeddings once for the whole list to save API quota
    r_embs = [get_gemini_embedding(s, api_key) for s in resume_skills]
    j_embs = [get_gemini_embedding(s, api_key) for s in jd_skills]
    sim = calculate_similarity_score(r_embs, j_embs)

    # 3. Behavioral Scores
    achieve = achievement_score(experience)
    own = ownership_score(experience)

    final_score = (0.35 * exact) + (0.30 * sim) + (0.20 * achieve) + (0.15 * own)

    return {
        "final_score": round(final_score, 3),
        "semantic_match": round(sim, 3),
        "keyword_match": round(exact, 3),
        "achievement_index": round(achieve, 3)
    }

In [10]:
!pip install google-generativeai
!pip install llama-index
!pip install llama-index-llms-gemini
!pip install llama-index-embeddings-google
!pip install langchain
!pip install langchain-google-genai
!pip install pydantic
from langchain_core.prompts import PromptTemplate

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

def generate_recruitment_report(results, evidence, jd, api_key):
    """Generates final recruiter-style summary using LangChain"""

    llm = ChatGoogleGenerativeAI(
        model="models/gemini-3.1-flash-lite-preview",
        google_api_key=api_key,
        temperature=0.3
    )

    template = """
    You are a Senior Technical Recruiter.

    Candidate Evaluation Scores:
    {results}

    Job Description:
    {jd}

    Resume Evidence:
    {evidence}

    Task:
    1. Write EXACTLY 3 concise sentences explaining the candidate's performance.
    2. Justify the achievement score and final score.
    3. Give a final verdict: Strong Match or Weak Match.
    """

    prompt = PromptTemplate.from_template(template)

    chain = prompt | llm

    response = chain.invoke({
        "results": results,
        "jd": jd,
        "evidence": evidence
    })

    return response.content

In [26]:
if __name__ == "__main__":

    # 1. SETUP
    import os
    from getpass import getpass

    API_KEY = getpass("Enter your Google API Key: ")
    FILE_PATH = "/content/resume.pdf"

    # 2. JOB DESCRIPTION
    jd = {
        "role": "Data Scientist",
        "company": "TechNova Analytics",
        "skills": ["Python", "Machine Learning", "Deep Learning", "Statistics", "SQL", "Data Visualization", "NLP", "TensorFlow", "PyTorch"],
        "requirements": [
            "Strong understanding of machine learning algorithms",
            "Experience with TensorFlow or PyTorch",
            "Experience with large datasets",
            "Strong statistical knowledge",
            "SQL and data pipelines experience",
            "Strong analytical thinking",
            "NLP is a plus"
        ],
        "responsibilities": ["Develop ML models", "Analyze data", "Work with teams", "Improve performance", "Communicate insights"],
        "experience_required": "2-5 years",
        "education": "Bachelor’s or Master’s in CS/Data Science"
    }

    # 3. PHASE 1 & 2: PARSING + SCORING
    print("--- PHASE 1 & 2: PARSING & SCORING ---")

    resume_data = parse_resume_gemini(FILE_PATH, API_KEY)
    results = evaluate_candidate_optimized(resume_data, jd, API_KEY)

    # 4. SIMPLIFIED EVIDENCE (NO LLAMAINDEX)
    print("--- PHASE 3: SIMPLE EVIDENCE EXTRACTION ---")

    raw_text = extract_raw_text(FILE_PATH)

    # Just pass relevant portion instead of full RAG
    evidence = raw_text[:1500]  # first 1500 chars (simple + works)

    # 5. LANGCHAIN REPORT
    print("--- PHASE 4: LANGCHAIN REPORTING ---")

    report = generate_recruitment_report(results, evidence, jd, API_KEY)

    print("\n" + "="*30)
    print("FINAL RECRUITMENT REPORT")
    print("="*30)
    print(report)

Enter your Google API Key: ··········
--- PHASE 1 & 2: PARSING & SCORING ---


--- PHASE 3: SIMPLE EVIDENCE EXTRACTION ---


--- PHASE 4: LANGCHAIN REPORTING ---

FINAL RECRUITMENT REPORT
[{'type': 'text', 'text': '1. The candidate demonstrates relevant technical exposure through a machine learning internship, but lacks the professional experience and educational credentials required for this role. \n2. The resume fails to meet the 2-5 year experience threshold and the specific degree requirements, resulting in an achievement index of 0.0. \n3. While the semantic match is high due to overlapping technical terminology, the significant gap in seniority and formal qualifications keeps the final score low.\n\n**Final Verdict: Weak Match**', 'extras': {'signature': 'EjQKMgG+Pvb7hF+M/vre8uXo9TZGN6pXMmaTbZF0J0K0jhTZ2SWoGREowFOn+5nPJuVxdLMJ'}}]
